# Export tree datasets to Parquet

Three datasets go out as **GeoParquet** (geometry preserved, readable by GeoPandas, DuckDB, QGIS 3.28+,
Apache Sedona):

| name | source | what it is |
|---|---|---|
| `city_public` | `references/Baeume_SFM_2026.gpkg` | Magdeburg tree registry — public green and street trees (85,302 points) |
| `city_private` | `references/Baeume_Liegenschaftsservice_2026.gpkg` | Magdeburg tree registry — municipal-property trees (5,665 points) |
| `trees_merged` | `data/orthophotos/segments/250m/ovgu_bbox_tcd_segformer_trees_merged.fgb` | pipeline output for `ovgu_bbox` at the 250 m tile size: crown polygons already enriched with Baumkataster heights and species |

The two registries are complementary, not overlapping — no Liegenschaftsservice point lies within 1 m of
an SFM point citywide, so loading both double-counts nothing.

Only the **250 m** merged layer is exported; the 100/500/1000 m variants exist but are alternative
tilings of the same area, and mixing them would double-count crowns.

Everything configurable is in the **Fields** cell — nothing below it needs editing.

## Fields

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "test_notebooks" else Path.cwd()

# --- fields ----------------------------------------------------------------
OUT_DIR     = ROOT / "data" / "parquet"   # where the .parquet files are written
TILE_SIZE_M = 250                         # which merged tiling to export (250 only, see above)
COMPRESSION = "zstd"                      # "snappy" | "zstd" | "gzip" | None
OVERWRITE   = True                        # False = leave existing .parquet files alone

SOURCES = {
    "city_public":  ROOT / "references" / "Baeume_SFM_2026.gpkg",
    "city_private": ROOT / "references" / "Baeume_Liegenschaftsservice_2026.gpkg",
    "trees_merged": ROOT / "data" / "orthophotos" / "segments" / f"{TILE_SIZE_M}m"
                         / "ovgu_bbox_tcd_segformer_trees_merged.fgb",
}
# ---------------------------------------------------------------------------

for name, src in SOURCES.items():
    assert src.exists(), f"input missing: {src}"
print(f"{len(SOURCES)} inputs found, writing to {OUT_DIR.relative_to(ROOT)}/")

3 inputs found, writing to data/parquet/


## Convert

`GeoDataFrame.to_parquet` writes GeoParquet: the geometry column is stored as WKB with the CRS carried
in the file metadata, so a round trip needs no reprojection or re-declaration.

In [2]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for name, src in SOURCES.items():
    dst = OUT_DIR / f"{name}.parquet"
    if dst.exists() and not OVERWRITE:
        print(f"skip   {name}  (exists, OVERWRITE=False)")
        continue

    gdf = gpd.read_file(src)
    gdf.to_parquet(dst, compression=COMPRESSION, index=False)

    rows.append({
        "name":      name,
        "rows":      len(gdf),
        "columns":   len(gdf.columns),
        "geom_type": gdf.geom_type.value_counts().idxmax(),
        "crs":       gdf.crs.to_string(),
        "src_MB":    round(src.stat().st_size / 1e6, 2),
        "parquet_MB": round(dst.stat().st_size / 1e6, 2),
    })
    print(f"wrote  {name}  ->  {dst.relative_to(ROOT)}")

summary = pd.DataFrame(rows)
summary["ratio"] = (summary["src_MB"] / summary["parquet_MB"]).round(1)
summary

wrote  city_public  ->  data/parquet/city_public.parquet
wrote  city_private  ->  data/parquet/city_private.parquet
wrote  trees_merged  ->  data/parquet/trees_merged.parquet


,name,rows,columns,geom_type,crs,src_MB,parquet_MB,ratio
0,city_public,85302,9,Point,EPSG:25832,17.06,1.91,8.9
1,city_private,5665,9,Point,EPSG:25832,3.49,0.13,26.8
2,trees_merged,1434,14,Polygon,EPSG:25832,3.07,0.72,4.3


## Verify the round trip

A conversion that silently drops rows, loses the CRS, or mangles geometry is worse than no conversion,
so read each file back and compare it against its source rather than trusting the write.

In [3]:
for name, src in SOURCES.items():
    a = gpd.read_file(src)
    b = gpd.read_parquet(OUT_DIR / f"{name}.parquet")

    assert len(a) == len(b), f"{name}: row count changed {len(a)} -> {len(b)}"
    assert list(a.columns) == list(b.columns), f"{name}: columns changed"
    assert a.crs == b.crs, f"{name}: CRS changed {a.crs} -> {b.crs}"
    assert a.geometry.geom_equals_exact(b.geometry, tolerance=0).all(), f"{name}: geometry changed"

    non_geom = [c for c in a.columns if c != a.geometry.name]
    pd.testing.assert_frame_equal(a[non_geom], b[non_geom], check_dtype=False)

    print(f"{name:14s} OK   {len(b):6,} rows | {len(b.columns):2d} cols | {b.crs.to_string()}")

city_public    OK   85,302 rows |  9 cols | EPSG:25832
city_private   OK    5,665 rows |  9 cols | EPSG:25832
trees_merged   OK    1,434 rows | 14 cols | EPSG:25832


## What is in each file

In [4]:
for name in SOURCES:
    g = gpd.read_parquet(OUT_DIR / f"{name}.parquet")
    print(f"\n=== {name} — {len(g):,} rows ===")
    print(g.dtypes.to_string())


=== city_public — 85,302 rows ===
Gattung lang           object
Stammumfang           float64
Baumhoehe             float64
Kronendurchmesser     float64
Pflanzjahr              int32
Objektbezeichnung      object
Objektart lang         object
baumnummer             object
geometry             geometry

=== city_private — 5,665 rows ===
Gattung lang           object
Baumnummer             object
Stammumfang           float64
Baumhoehe             float64
Kronendurchmesser     float64
Pflanzjahr              int32
Objektbezeichnung      object
Objektart lang         object
geometry             geometry



=== trees_merged — 1,434 rows ===
tree_id                      int32
height_m                   float64
allometric_height_m        float64
crown_radius_m             float64
crown_area_m2              float64
vegetation_model            object
species                     object
height_source               object
is_deciduous                object
trunk_circumference_cm      object
planting_year               object
bk_match_dist_m             object
h_global_m                 float64
geometry                  geometry


## Reading them back

```python
import geopandas as gpd
trees = gpd.read_parquet("data/parquet/trees_merged.parquet")     # full layer, geometry intact
```

Parquet is columnar, so a subset of fields can be read without touching the rest of the file — useful
on `city_public`, where most analyses need three columns out of nine:

```python
import pandas as pd
h = pd.read_parquet("data/parquet/city_public.parquet",
                    columns=["Gattung lang", "Baumhoehe", "Kronendurchmesser"])
```

Note `pd.read_parquet` returns a plain DataFrame with no geometry; use `gpd.read_parquet` when the
points or polygons are needed.